In [ ]:
!pip install -q open_clip_torch

实验：一图三文字

In [ ]:
import torch
import open_clip
from PIL import Image
import requests

# 1. 加载 CLIP 模型
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model.eval()

# 2. 下载一张测试图片
url = "https://images.unsplash.com/photo-1552053831-71594a27632d?w=640"
image = Image.open(requests.get(url, stream=True, timeout=20).raw).convert("RGB")

# 3. 三句候选描述
texts = [
    "a photo of a dog",
    "a photo of a cat",
    "a photo of a car"
]

# 4. 图片和文本分别转成向量
image_input = preprocess(image).unsqueeze(0)
text_input = tokenizer(texts)

with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_input)

    # 归一化
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # 计算相似度并转成概率
    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)

# 5. 输出结果
for text, score in zip(texts, similarity[0]):
    print(f"{text}: {score.item():.4f}")

实验：十图十文字

In [ ]:
# 定义设备
device = "cuda" if torch.cuda.is_available() else "cpu"

# 把已经加载好的模型移动到对应设备
model = model.to(device)

In [ ]:
# 每张图片对应一句正确描述
# 列表顺序就是正确答案的索引
samples = [
    ("a photo of a dog",       "https://images.unsplash.com/photo-1552053831-71594a27632d?w=640"),
    ("a photo of a cat",       "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=640"),
    ("a photo of a car",       "https://images.unsplash.com/photo-1503376780353-7e6692767b70?w=640"),
    ("a photo of an airplane", "https://images.unsplash.com/photo-1436491865332-7a61a109cc05?w=640"),
    ("a photo of a bicycle",   "https://images.unsplash.com/photo-1485965120184-e220f721d03e?w=640"),
    ("a cup of coffee",        "https://images.unsplash.com/photo-1509042239860-f550ce710b93?w=640"),
    ("a pizza on a table",     "https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=640"),
    ("a laptop on a desk",     "https://images.unsplash.com/photo-1496181133206-80ce9b88a853?w=640"),
    ("a field of flowers",     "https://images.unsplash.com/photo-1490750967868-88aa4486c946?w=640"),
    ("a stack of books",       "https://images.unsplash.com/photo-1544947950-fa07a98d237f?w=640"),
]

images = []
captions = []

# 下载并预处理图片
for caption, url in samples:
    try:
        # 输入：图片网址
        # 输出：RGB 图片
        image = Image.open(
            requests.get(url, stream=True, timeout=20).raw
        ).convert("RGB")

        # 预处理成模型需要的图片张量
        images.append(preprocess(image))

        # 保存对应文字
        captions.append(caption)

        print("加载成功:", caption)

    except Exception as e:
        print("加载失败:", caption)
        print("原因:", e)

# 把所有图片堆成一个批次
# 形状大约：[图片数量, 3, 224, 224]
image_input = torch.stack(images).to(device)

# 把所有文字转成 token
text_input = tokenizer(captions).to(device)

# 不训练，只做推理
with torch.no_grad():
    # 图片 -> 图片向量
    image_features = model.encode_image(image_input)

    # 文字 -> 文字向量
    text_features = model.encode_text(text_input)

    # 归一化
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # 相似度矩阵
    # 行：图片
    # 列：文字
    similarity = image_features @ text_features.T

# 每张图片选择最匹配的文字
predicted_indices = similarity.argmax(dim=1)

# 正确答案就是图片自己的索引
correct_indices = torch.arange(len(captions), device=device)

# 统计正确数量
correct_count = (predicted_indices == correct_indices).sum().item()
accuracy = correct_count / len(captions)

print("\n========== 结果 ==========")

for i, caption in enumerate(captions):
    pred_index = predicted_indices[i].item()
    predicted_caption = captions[pred_index]
    result = "正确" if i == pred_index else "错误"

    print(
        f"图片 {i + 1}: "
        f"正确文字 = {caption} | "
        f"预测文字 = {predicted_caption} | "
        f"{result}"
    )

print("\nTop-1 准确率:", accuracy)
print("正确数量:", correct_count, "/", len(captions))